# Run LASSQD

This guide builds a twelve-hydrogen molecule, prepares localized active orbitals, and
runs **LASSQD**: sample-based quantum diagonalization (SQD) as the fragment solver
for localized active space self-consistent field (LASSCF). Everything needed to
construct the system is below. Sampling runs locally with `ffsim.qiskit.FfsimSampler`
and 10% global depolarizing noise to improve configuration coverage.

We use two H₆ fragments, each with six electrons in six spatial orbitals.
Each fragment is a singlet with three alpha and three beta electrons, giving
six alpha and six beta electrons in total. Each fragment has a separate
12-qubit circuit, simulated in its fixed electron-number sector with
400 state amplitudes.
The small system also lets us compare against classical LASSCF with exact
fragment solvers.

Run the code cells in order in a fresh Python kernel. Runtime and numerical
details depend on your machine and package versions. With the settings below,
a checked run converged in three cycles to about **−6.513062 Hartree**; the
classical LASSCF reference is about **−6.513063 Hartree**.

## Imports and the hybrid workflow

Each hybrid cycle performs the following steps:

1. Build an effective Hamiltonian for each fragment at the current orbitals.
2. Prepare a **local unitary cluster Jastrow (LUCJ)** circuit per fragment,
   initialized by compressed double factorization of CCSD amplitudes. Submit
   the fragments as separate measured circuits in one sampler job.
3. Use each fragment's measured bitstrings to select determinant subspaces.
   SQD diagonalizes the fragment Hamiltonian in these subspaces and returns
   energies and reduced density matrices (RDMs).
4. Use those RDMs for one LASSCF orbital step, then repeat with the updated orbitals.

`run_lassqd` coordinates these steps. We use its default `lucj_circuit` builder
and `FfsimSampler`.

In [1]:
import time
from functools import partial
from math import comb

import numpy as np
from ffsim.qiskit import FfsimSampler
from pyscf import gto, scf
from qiskit_addon_sqd.fermion import solve_sci_batch

from lassqd import FragmentSQD, LASSCFNoSymm, run_lassqd

rng = np.random.default_rng(12345)

## Build H₁₂ and run Hartree–Fock

The atoms lie on the x-axis. Each H₆ group has 1.0 Å nearest-neighbor spacing,
and the closest atoms belonging to different groups are 2.0 Å apart.
STO-3G supplies one spatial basis function per hydrogen, giving twelve orbitals
for twelve electrons. All twelve orbitals will be active, so there are no inactive
core or external virtual orbitals in this example.

PySCF's molecular `spin` is the alpha-minus-beta electron count, or twice the spin
projection. We use `spin=0` and a restricted Hartree–Fock (RHF) calculation to
generate the initial orbitals. The fragment spin assignments are defined
separately in the next section.

In [2]:
x_positions = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0]
mol = gto.M(
    atom=[("H", (x, 0.0, 0.0)) for x in x_positions],
    unit="Angstrom",
    basis="sto-6g",
    charge=0,
    spin=0,
    verbose=0,
)
mf = scf.RHF(mol).run()
assert mf.converged, "RHF did not converge; inspect the mean-field calculation."

print(f"Electrons: {mol.nelectron}; spatial orbitals: {mf.mo_coeff.shape[1]}")
print(f"RHF total energy: {mf.e_tot:.10f} Hartree")

Electrons: 12; spatial orbitals: 12
RHF total energy: -6.3113494154 Hartree


## Define the active fragments and localize the orbitals

Atom indices are **zero-based**. `ncas_sub` counts spatial orbitals per fragment;
`nelecas_sub` gives each fragment's `(alpha, beta)` electron counts.
`spin_sub` contains spin **multiplicities** $2S+1$, so both H₆ singlets use 1.

| Fragment | Atom indices | Active orbitals | Alpha electrons | Beta electrons | Multiplicity |
| --- | --- | --- | --- | --- | --- |
| 0 | 0, 1, 2, 3, 4, 5 | 6 | 3 | 3 | 1 |
| 1 | 6, 7, 8, 9, 10, 11 | 6 | 3 | 3 | 1 |

Both fragments target $S=0$ and have spin projection $M_S=0$. The product of
two singlet fragment states is also a total-spin singlet. We set the SQD spin
target explicitly below, in addition to defining the fragment electron counts.

`localize_init_guess` assigns active orbitals to the atom groups and returns
orthonormal orbitals ordered by fragment. Here the entire orbital space is active,
so no orbital sorting or external orbital files are needed. For a larger basis,
select the active orbitals before localization.

In [3]:
fragment_atoms = ([0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11])
ncas_sub = (6, 6)
nelecas_sub = ((3, 3), (3, 3))
spin_sub = (1, 1)

las = LASSCFNoSymm(mf, ncas_sub, nelecas_sub, spin_sub=spin_sub)
mo_initial = las.localize_init_guess(fragment_atoms, mf.mo_coeff)

np.testing.assert_allclose(
    mo_initial.T @ mf.get_ovlp() @ mo_initial,
    np.eye(mo_initial.shape[1]),
    atol=1e-10,
    rtol=0,
)
print(f"Fragments: {las.nfrags}; active orbitals: {las.ncas}; core orbitals: {las.ncore}")
print(f"Qubits per fragment: {[2 * n for n in ncas_sub]}")

Fragments: 2; active orbitals: 12; core orbitals: 0
Qubits per fragment: [12, 12]


## Compute a classical LASSCF reference

Use a separate LAS object with the same fragment definition and a copy of the
same initial orbitals. Its default classical fragment solvers provide a convenient
reference without sampling. This step is inexpensive for these H₆ fragments
and is used only to assess the result; LASSQD starts from `mo_initial`, not the
optimized reference.

This reference is exact **within each fragment**. The LAS product form still
restricts the molecular wave function, so it is not a full-molecule FCI reference.

In [4]:
las_reference = LASSCFNoSymm(mf, ncas_sub, nelecas_sub, spin_sub=spin_sub)
las_reference.kernel(mo_initial.copy())
assert las_reference.converged, "The classical LASSCF reference did not converge."
e_reference = float(las_reference.e_tot)
print(f"Classical LASSCF total energy: {e_reference:.10f} Hartree")

Classical LASSCF total energy: -6.5130626352 Hartree


## Configure SQD and local circuit sampling

Create one `FragmentSQD` instance per fragment. The main controls are:

| Setting | Here | Meaning |
| --- | --- | --- |
| `max_iterations` | 2 | Maximum configuration-recovery rounds per fragment solve |
| `num_batches` | 2 | Independently subsampled batches per round |
| `samples_per_batch` | 1,000 | Number of sampled bitstring rows requested per batch, limited by available rows |
| `shots` | 4,096 | Measurements per fragment per hybrid cycle |
| `global_depolarizing` | 0.1 | Probability of replacing a measurement with a uniformly random bitstring |
| `max_cycles` | 20 | Maximum number of sampling/orbital-update cycles |
| `conv_tol` | $10^{-6}$ Hartree | Threshold on the change in total energy between cycles |

The solver selects the lowest-energy batch across all recovery rounds. On the
first round it postselects for the requested alpha and beta electron counts;
subsequent rounds use the best batch's occupations for configuration recovery.
Recovery can stop early when both the energy and occupation tolerances are met.
We explicitly disable determinant carryover with `carryover_threshold=None`
and request no SQD output directory. The batch solver targets the singlet
value $S(S+1)=0$ for each H₆ fragment and sets the Davidson iteration
limit and tolerance.

`FfsimSampler` simulates the native ffsim gates in each fragment's fixed
alpha/beta electron-number sector. Use `glue_circuits=False` in `run_lassqd` to
submit the two measured fragment circuits separately in one job. The sampler
infers each fragment's orbital and electron counts from its Hartree–Fock
preparation gate. No pass manager is needed: keep the native ffsim gates intact.

Set `global_depolarizing=0.1` to replace each measurement with probability 10%
by a uniformly random 12-bit string. This gives configurations with negligible
circuit probability a chance to appear. The noise also generates strings with
incorrect alpha or beta electron counts, which SQD handles through postselection
and configuration recovery. The added samples broaden the selected subspace;
SQD still computes its energies from the fragment Hamiltonian.

The `rng` initialized above is shared by the sampler and all SQD solvers. Each
use advances the same random stream. Restarting the kernel and running all cells
reproduces the stream, but changing the order or number of random draws affects
later steps. Different package versions can also change results.

In [5]:
solvers = [
    FragmentSQD(
        samples_per_batch=1000,
        max_iterations=2,
        num_batches=2,
        carryover_threshold=None,
        sci_solver=partial(solve_sci_batch, spin_sq=0.0),
        seed=rng,
        verbose=0,
    )
    for _ in range(las.nfrags)
]

sampler = FfsimSampler(seed=rng, global_depolarizing=0.1)

shots = 4096
max_cycles = 20
conv_tol = 1e-6

## Run the hybrid calculation

`run_lassqd` fills each solver's `counts`, installs the SQD fragment kernels on
`las`, and alternates sampling with orbital updates. Its default circuit builder
uses one LUCJ layer. The callback prints a compact progress line after each cycle;
the callback's cycle index starts at zero.

The call updates `las` and the solver objects in place. To repeat this calculation
from the same starting point and RNG state, restart the kernel and run all
cells. Re-executing only this cell reuses mutable solver and LAS state.

In [6]:
start_time = time.perf_counter()


def report_cycle(cycle, current_las):
    elapsed = time.perf_counter() - start_time
    print(
        f"Cycle {cycle + 1:2d}: E = {current_las.e_tot:.10f} Hartree"
        f"  ({elapsed:.1f} s elapsed)"
    )


result = run_lassqd(
    las,
    mo_initial.copy(),
    solvers,
    sampler,
    glue_circuits=False,
    shots=shots,
    max_cycles=max_cycles,
    conv_tol=conv_tol,
    callback=report_cycle,
)
print(f"Converged: {result.converged}; cycles: {len(result.e_hist)}")

Cycle  1: E = -6.5130401658 Hartree  (2.1 s elapsed)
Cycle  2: E = -6.5130620262 Hartree  (3.2 s elapsed)
Cycle  3: E = -6.5130624405 Hartree  (4.2 s elapsed)
Converged: True; cycles: 3


## Inspect energies and convergence

`result.e_tot` is the molecular total energy, including nuclear repulsion.
`result.e_hist` contains one such energy per hybrid cycle. These are the energies
to compare with classical LASSCF; individual fragment SQD histories use fragment
Hamiltonians and are not molecular total energies.

A checked run gave LASSQD **−6.5130624411 Hartree** and classical LASSCF
**−6.5130626352 Hartree**, a difference of about 1.9 × 10⁻⁷ Hartree.
Treat these as illustrative values. `result.converged` means only that the last
absolute energy change is smaller than `conv_tol`; it does not certify orbital
gradient convergence or accuracy relative to an exact molecular solution.

Each fragment's fixed-electron determinant space has
$\binom{6}{3}\binom{6}{3}=400$ determinants. Compare the final SQD subspace
dimensions printed below with this full-space dimension. SQD forms subspaces
from combinations of selected alpha and beta strings, so full subspace coverage
does not require observing all 400 valid bitstrings directly. The larger batch
size and 10% depolarizing noise help broaden configuration coverage. Finite
sampling makes coverage probabilistic; these settings do not guarantee full
coverage in every run or in larger spaces.

The current implementation uses only the alpha one-electron Hamiltonian
`h1s[0]` for both circuit construction and SQD, even when the embedding Hamiltonian
is spin-dependent. Exact agreement with classical LASSCF is therefore not a
general guarantee for spin-polarized fragments.

In [7]:
print("Cycle       Total energy (Hartree)       Delta E (Hartree)")
for cycle, energy in enumerate(result.e_hist, start=1):
    delta = "--" if cycle == 1 else f"{energy - result.e_hist[cycle - 2]:+.3e}"
    print(f"{cycle:5d}       {energy: .10f}             {delta:>10}")

energy_difference = float(result.e_tot - e_reference)
print(f"LASSQD total energy:           {result.e_tot:.10f} Hartree")
print(f"Classical LASSCF total energy: {e_reference:.10f} Hartree")
print(f"LASSQD minus reference:        {energy_difference:+.3e} Hartree")

assert np.isfinite(result.e_hist).all(), "The energy history contains a nonfinite value."
assert result.converged, "Increase max_cycles and inspect the energy changes before continuing."
assert abs(result.e_hist[-1] - result.e_hist[-2]) < conv_tol

for ifrag, solver in enumerate(solvers):
    norb = ncas_sub[ifrag]
    nalpha, nbeta = nelecas_sub[ifrag]
    full_dimension = comb(norb, nalpha) * comb(norb, nbeta)
    print(
        f"Fragment {ifrag}: final-round SQD subspace dimensions "
        f"{solver.d_hist[-1].astype(int).tolist()} (full space: {full_dimension})"
    )

Cycle       Total energy (Hartree)       Delta E (Hartree)
    1       -6.5130401658                     --
    2       -6.5130620262             -2.186e-05
    3       -6.5130624405             -4.143e-07
LASSQD total energy:           -6.5130624405 Hartree
Classical LASSCF total energy: -6.5130626352 Hartree
LASSQD minus reference:        +1.946e-07 Hartree
Fragment 0: final-round SQD subspace dimensions [400, 400] (full space: 400)
Fragment 1: final-round SQD subspace dimensions [400, 400] (full space: 400)


## Check electron counts and the returned wave function

Keep `result.mo_coeff`, `result.casdm1frs`, and `result.casdm2fr` together: the RDMs
are expressed in the active orbitals of these returned coefficients.
Combining them with the initial orbitals would describe a different wave function.

For fragment `i`, the spin-separated 1-RDM has shape
`(nroots, 2, n_i, n_i)` and the spin-summed 2-RDM has shape
`(nroots, n_i, n_i, n_i, n_i)`. This calculation has one root and six orbitals
per fragment. Tracing each spin block of root zero should recover `(3, 3)` for
both fragments.

Finally, evaluate the energy from the returned orbitals and RDMs to check that
they are consistent with `result.e_tot`.

In [8]:
for ifrag, (dm1, dm2, nelec) in enumerate(
    zip(result.casdm1frs, result.casdm2fr, nelecas_sub)
):
    norb = ncas_sub[ifrag]
    assert dm1.shape == (1, 2, norb, norb)
    assert dm2.shape == (1, norb, norb, norb, norb)
    assert np.isfinite(dm1).all() and np.isfinite(dm2).all()
    populations = np.trace(dm1[0], axis1=-2, axis2=-1)
    np.testing.assert_allclose(populations, nelec, atol=1e-7, rtol=0)
    print(
        f"Fragment {ifrag}: 1-RDM {dm1.shape}, 2-RDM {dm2.shape}; "
        f"N_alpha = {populations[0]:.6f}, N_beta = {populations[1]:.6f}"
    )

energy_from_rdms = las.energy_nuc() + las.energy_elec(
    mo_coeff=result.mo_coeff,
    casdm1frs=result.casdm1frs,
    casdm2fr=result.casdm2fr,
)
np.testing.assert_allclose(energy_from_rdms, result.e_tot, atol=1e-8, rtol=0)
print(f"Energy reconstructed from orbitals and RDMs: {energy_from_rdms:.10f} Hartree")

Fragment 0: 1-RDM (1, 2, 6, 6), 2-RDM (1, 6, 6, 6, 6); N_alpha = 3.000000, N_beta = 3.000000
Fragment 1: 1-RDM (1, 2, 6, 6), 2-RDM (1, 6, 6, 6, 6); N_alpha = 3.000000, N_beta = 3.000000
Energy reconstructed from orbitals and RDMs: -6.5130624405 Hartree


## Adapt the example

- **Geometry and basis:** Change the coordinates in Å and the PySCF basis, then
  recompute RHF and localize new initial orbitals. A larger basis introduces
  additional orbitals; choose an active space rather than assuming all orbitals
  belong to these two six-orbital fragments.
- **Active spaces and spins:** Update the zero-based atom groups, `ncas_sub`,
  `(alpha, beta)` electron counts, and multiplicities together. Account for
  doubly occupied inactive core orbitals in the molecular electron count. Each
  fragment needs two qubits per active spatial orbital. For a selected active
  subset, `las.sort_mo` uses **one-based orbital indices by default**; place the
  selected orbitals in the active block before calling `localize_init_guess`.
- **Sampling and SQD cost:** More shots improve the chance of observing rare
  configurations. Larger batches expand the selected determinant space, while
  more batches and recovery iterations repeat the classical solves. These
  controls increase cost in different ways; inspect the energy history and SQD
  subspace dimensions as you change them.
- **Convergence:** If the cycle cap is reached while energies are still changing,
  increase `max_cycles`. If the energy fluctuates because of finite sampling,
  increase sampling or batch coverage and assess the fluctuations before choosing
  `conv_tol`. Tighter energy thresholds alone do not guarantee a more accurate
  wave function.

After making changes, restart the kernel and run all cells to rebuild the
molecule, reference, fragment solvers, and sampler consistently. The larger
[FeFe examples](../examples/README.md) illustrate other systems and execution
settings once this local workflow is familiar.